# 03 - Modele Regression Logistique

## 1. Objectif et protocole

Etablir la baseline probabiliste principale et documenter ses performances confirmatoires.

Ce notebook genere des artefacts de preuve pour le rapport. Il ne contient pas de conclusion;
les resultats tuned sont confirmes uniquement par nested CV, tandis que les gaps train/validation
restent reserves aux variantes non tunees en simple CV.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sns.set_theme(style="whitegrid", context="talk")
pd.options.display.float_format = lambda value: f"{value:.4f}"

from src import (
    calculer_importance_permutation_exploratoire,
    calculer_metriques_classes_depuis_predictions,
    calculer_top_k_accuracies,
    charger_donnees,
    construire_matrice_confusion,
    construire_tableau_gaps,
    construire_tableau_plis_externes,
    construire_tableau_variantes,
    creer_notebook_context,
    evaluer_variantes_untuned,
    executer_tuning_confirmatoire,
    tracer_courbes_precision_rappel_ovr,
    tracer_heatmap_tuning,
    tracer_histogramme_confiance,
    tracer_matrice_confusion,
    tracer_metrique_par_classe,
    tracer_reliability_diagram,
    tracer_top_confusions,
)
from src.evaluation import (
    calculer_metriques,
    calculer_statistiques_confiance,
    evaluer_dummy_classifier,
    extraire_top_confusions,
)
from src.notebook_support import NOTEBOOK_MODEL_SPECS
from src.resultats import (
    MetricsArtifactValidationError,
    charger_tableau_mesures_valides,
    filtrer_legacy,
    normaliser_stades,
)


In [ ]:
MODEL_NAME = "regression_logistique"
context = creer_notebook_context(MODEL_NAME)
spec = context.spec
model_definition = context.definition
X, y, label_encoder = context.X, context.y, context.label_encoder

protocole = pd.DataFrame(
    [
        {
            "modele": model_definition["nom_affiche"],
            "famille": model_definition["famille"],
            "description": model_definition["description"],
            "timing_policy": context.timing_policy,
            "cv_externe": context.cv_eval.get_n_splits(),
            "cv_interne": context.cv_tuning.get_n_splits(),
        }
    ]
)
display(protocole)


            ## 2. Design experimental specifique

            - La regression logistique est la reference lineaire probabiliste du projet.
- Le scaling est obligatoire pour un conditionnement numerique stable.
- Les probabilites out-of-fold servent a documenter calibration, top-k et confiance predictive.


In [ ]:
design_table = pd.DataFrame(
    {
        "variante": [variant.name for variant in spec.untuned_variants],
        "type": [variant.kind for variant in spec.untuned_variants],
        "parametres_forces": [variant.params or {} for variant in spec.untuned_variants],
    }
)
display(design_table)


## 3. Resultats des variantes non tunees

Cette section mesure les variantes brutes et preprocessees en simple CV. Les tableaux affichent la
performance moyenne, la dispersion inter-plis, puis les gaps train/validation uniquement pour ces
variantes non tunees.


In [ ]:
resultats_untuned = evaluer_variantes_untuned(context)
tableau_untuned = pd.DataFrame(resultats_untuned).sort_values("val_f1_macro_mean", ascending=False)
display(tableau_untuned.round(4))
display(construire_tableau_gaps(resultats_untuned))


## 4. Resultat confirmatoire tune

Le tuning exploratoire identifie une zone de travail utile, puis la performance finale publiee est
estimee avec nested CV. Les metriques du tableau ci-dessous servent de reference confirmatoire.


In [ ]:
tuning = executer_tuning_confirmatoire(context)
tableau_variantes = construire_tableau_variantes(resultats_untuned, tuning["metrics"])
tableau_plis = construire_tableau_plis_externes(tuning["nested"])

display(tableau_variantes.round(4))
display(tableau_plis.round(4))


## 5. Diagnostics du modele

Les graphiques suivants montrent la comparaison entre variantes, la variabilite inter-plis du resultat
tuned, puis l'analyse d'erreurs sur les predictions out-of-fold du schema confirmatoire.


In [ ]:
def tracer_dot_whisker(tableau, categorie, moyenne, dispersion, titre, orientation="vertical"):
    donnees = tableau.sort_values(moyenne, ascending=(orientation == "horizontal")).copy()
    figure, axe = plt.subplots(figsize=(10.5, 5.8))
    if orientation == "horizontal":
        axe.errorbar(
            donnees[moyenne],
            donnees[categorie],
            xerr=donnees[dispersion],
            fmt="o",
            color="#0b1f3a",
            ecolor="#4C78A8",
            capsize=4,
            markersize=7,
        )
        axe.set_xlabel(moyenne.replace("_", " "), fontweight="bold")
        axe.set_ylabel(categorie.replace("_", " "), fontweight="bold")
    else:
        axe.errorbar(
            donnees[categorie],
            donnees[moyenne],
            yerr=donnees[dispersion],
            fmt="o",
            color="#0b1f3a",
            ecolor="#4C78A8",
            capsize=4,
            markersize=7,
        )
        axe.set_xlabel(categorie.replace("_", " "), fontweight="bold")
        axe.set_ylabel(moyenne.replace("_", " "), fontweight="bold")
        axe.tick_params(axis="x", rotation=30)
    axe.set_title(titre)
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_barres_stades(tableau, titre):
    figure, axe = plt.subplots(figsize=(10.8, 6.0))
    sns.barplot(
        data=tableau,
        x="modele",
        y="val_f1_macro_mean",
        hue="stade",
        palette="crest",
        ax=axe,
    )
    axe.set_title(titre)
    axe.set_xlabel("Modele", fontweight="bold")
    axe.set_ylabel("Validation F1-Macro", fontweight="bold")
    axe.tick_params(axis="x", rotation=30)
    sns.move_legend(axe, "upper left", bbox_to_anchor=(1.02, 1), frameon=False, title="Stade")
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_barres_gains(tableau, colonne, titre):
    donnees = tableau.sort_values(colonne, ascending=False).copy()
    figure, axe = plt.subplots(figsize=(10.2, 5.6))
    sns.barplot(data=donnees, x="modele", y=colonne, palette="mako", ax=axe)
    axe.axhline(0.0, color="#222222", linewidth=1.0)
    axe.set_title(titre)
    axe.set_xlabel("Modele", fontweight="bold")
    axe.set_ylabel(colonne.replace("_", " "), fontweight="bold")
    axe.tick_params(axis="x", rotation=30)
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


def tracer_points_plis(tableau, colonne, titre):
    donnees = tableau.sort_values("fold").copy()
    figure, axe = plt.subplots(figsize=(9.6, 5.4))
    sns.pointplot(data=donnees, x="fold", y=colonne, color="#0b1f3a", ax=axe)
    axe.set_title(titre)
    axe.set_xlabel("Pli externe", fontweight="bold")
    axe.set_ylabel(colonne.replace("_", " "), fontweight="bold")
    sns.despine(ax=axe)
    figure.tight_layout()
    return axe


In [ ]:
tracer_dot_whisker(
    tableau_variantes,
    categorie="variante",
    moyenne="val_f1_macro_mean",
    dispersion="val_f1_macro_std",
    titre="Comparaison des variantes sur F1-macro",
)
plt.show()

tracer_points_plis(
    tableau_plis,
    colonne="val_f1_macro",
    titre="Variabilite des plis externes (F1-macro)",
)
plt.show()


In [ ]:
predictions_tuned = tuning["nested"]["oof_predictions"].copy()
confusions = extraire_top_confusions(
    predictions_tuned["y_true"].to_numpy(),
    predictions_tuned["y_pred"].to_numpy(),
    label_encoder=label_encoder,
    top_n=12,
)
metriques_classes = calculer_metriques_classes_depuis_predictions(
    predictions_tuned,
    label_encoder=label_encoder,
)
matrice_confusion = construire_matrice_confusion(
    predictions_tuned["y_true"].to_numpy(),
    predictions_tuned["y_pred"].to_numpy(),
    label_encoder=label_encoder,
    normalize="true",
)

display(confusions)
display(metriques_classes.sort_values(["recall", "f1"]).head(15).round(4))
tracer_top_confusions(confusions, top_n=12, titre="Paires de confusion les plus frequentes")
plt.show()
tracer_metrique_par_classe(
    metriques_classes,
    metrique="recall",
    top_n=15,
    ordre="asc",
    titre="Classes les plus difficiles",
)
plt.show()
tracer_matrice_confusion(
    matrice_confusion,
    titre="Matrice de confusion normalisee (appendice)",
    annot=False,
    rotation_x=90,
)
plt.show()


### Diagnostic exploratoire du tuning

Cette section reste exploratoire. Elle visualise la surface de recherche du grid search
interne, sans lui attribuer une valeur confirmatoire superieure a la nested CV.


In [ ]:
grid_results = tuning["grid_results"].copy()
x_axis, y_axis = spec.tuning_heatmap_axes
facet = spec.tuning_facet_column

if facet and facet in grid_results.columns:
    for facet_value in sorted(grid_results[facet].astype(str).unique()):
        subset = grid_results[grid_results[facet].astype(str) == str(facet_value)].copy()
        tracer_heatmap_tuning(
            subset,
            x=x_axis,
            y=y_axis,
            valeur="mean_test_f1_macro",
            titre=f"Surface de tuning exploratoire - {facet}={facet_value}",
        )
        plt.show()
else:
    tracer_heatmap_tuning(
        grid_results,
        x=x_axis,
        y=y_axis,
        valeur="mean_test_f1_macro",
        titre="Surface de tuning exploratoire",
    )
    plt.show()


### Diagnostics probabilistes confirmatoires

Les probabilites ci-dessous proviennent des predictions out-of-fold du schema nested CV.
Elles documentent calibration, top-k et confiance predictive sans re-utiliser un refit global
comme preuve principale.


In [ ]:
probabilites_tuned = tuning["nested"]["oof_probabilities"].copy()
top_k = calculer_top_k_accuracies(probabilites_tuned, ks=(3, 5))
confiance = calculer_statistiques_confiance(probabilites_tuned)

from sklearn.metrics import log_loss

logloss = log_loss(
    probabilites_tuned["y_true"],
    probabilites_tuned.filter(like="proba__"),
    labels=sorted(probabilites_tuned["y_true"].unique()),
)

display(pd.DataFrame([{"log_loss": logloss, **top_k, **confiance}]).round(4))
tracer_histogramme_confiance(probabilites_tuned, titre="Distribution de confiance OOF")
plt.show()
tracer_reliability_diagram(probabilites_tuned, titre="Reliability diagram OOF")
plt.show()
tracer_courbes_precision_rappel_ovr(probabilites_tuned, titre="Precision-rappel OvR (OOF)")
plt.show()


## 6. Resume des artefacts exportes

Cette cellule recense les artefacts produits ou mis a jour par le notebook pendant l'execution.


In [ ]:
artefacts = pd.DataFrame(
    [
        {"artefact": "metrics_tuned", "path": str(tuning["metrics_path"])},
        {"artefact": "predictions_tuned", "path": str(tuning["predictions_path"])},
        {"artefact": "grid_search", "path": str(tuning["grid_path"])},
        {"artefact": "probabilities_tuned", "path": str(tuning["probabilities_path"]) if tuning["probabilities_path"] else "n/a"},
    ]
)
display(artefacts)
